## Model Training 

In [2]:
import pandas as pd
import pickle

In [3]:
# Load DataSet

data = pd.read_csv(rf'Churn_Modelling.csv')
data

,RowNumber,CustomerId,Surname,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,1,15634602,Hargrave,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,2,15647311,Hill,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,3,15619304,Onio,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,4,15701354,Boni,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,5,15737888,Mitchell,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,9996,15606229,Obijiaku,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,9997,15569892,Johnstone,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,9998,15584532,Liu,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,9999,15682355,Sabbatini,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [4]:
# Preprocessing
# Droppping Irrelevent Columns

data = data.drop(columns=['RowNumber', 'CustomerId', 'Surname'])
data

,CreditScore,Geography,Gender,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited
0,619,France,Female,42,2,0.00,1,1,1,101348.88,1
1,608,Spain,Female,41,1,83807.86,1,0,1,112542.58,0
2,502,France,Female,42,8,159660.80,3,1,0,113931.57,1
3,699,France,Female,39,1,0.00,2,0,0,93826.63,0
4,850,Spain,Female,43,2,125510.82,1,1,1,79084.10,0
...,...,...,...,...,...,...,...,...,...,...,...
9995,771,France,Male,39,5,0.00,2,1,0,96270.64,0
9996,516,France,Male,35,10,57369.61,1,1,1,101699.77,0
9997,709,France,Female,36,7,0.00,1,0,1,42085.58,1
9998,772,Germany,Male,42,3,75075.31,2,1,0,92888.52,1


In [5]:
# Applying OneHot Encoding

from sklearn.preprocessing import OneHotEncoder

ohe_geography = OneHotEncoder(sparse_output = True)
ohe_gender = OneHotEncoder(sparse_output = True)

geo_encoder = ohe_geography.fit_transform(data[['Geography']])
geo_encoder_df = pd.DataFrame(geo_encoder.toarray(), columns=ohe_geography.get_feature_names_out())

gender_encoder = ohe_gender.fit_transform(data[['Gender']])
gen_encoder_df = pd.DataFrame(gender_encoder.toarray(), columns=ohe_gender.get_feature_names_out())

data = pd.concat([data, geo_encoder_df, gen_encoder_df], axis=1).drop(columns=['Geography', 'Gender']) 

In [6]:
data.head()

,CreditScore,Age,Tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Exited,Geography_France,Geography_Germany,Geography_Spain,Gender_Female,Gender_Male
0,619,42,2,0.00,1,1,1,101348.88,1,1.0,0.0,0.0,1.0,0.0
1,608,41,1,83807.86,1,0,1,112542.58,0,0.0,0.0,1.0,1.0,0.0
2,502,42,8,159660.80,3,1,0,113931.57,1,1.0,0.0,0.0,1.0,0.0
3,699,39,1,0.00,2,0,0,93826.63,0,1.0,0.0,0.0,1.0,0.0
4,850,43,2,125510.82,1,1,1,79084.10,0,0.0,0.0,1.0,1.0,0.0


In [7]:
#  Divide Data Into Dependent And Independent Features
X = data.drop(columns=['Exited'])
y = data[['Exited']]


# Splitting Data in Training And Test Dataset
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=786)

# Scaling Features
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


In [8]:
# Save The Encoders

with open('onehot_encoder_geography.pkl', 'wb') as file:
    pickle.dump(ohe_geography, file)

with open('onehot_encoder_gender.pkl', 'wb') as file:
    pickle.dump(ohe_gender, file)

with open ('standard_scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

ANN Implementation

In [9]:
# Build ANN Model

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

model = Sequential([
    Dense(64, activation='relu', input_shape=(X_train.shape[1],)), # Hidden Layer 1
    Dense(32, activation='relu'),  # Hidden Layer 2
    Dense(1, activation='sigmoid')  # Output Layer
])

d:\DS_Work\DLL\Customer Churn Model\venv\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [10]:
# Compile The Model

import tensorflow as tf

opt = tf.keras.optimizers.Adam(learning_rate=0.01) 
loss = tf.keras.losses.BinaryCrossentropy()
model.compile(optimizer=opt, loss=loss, metrics=['accuracy'])

In [11]:
# Setup the Tensorboard

from datetime import datetime
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard

log_dir = f'logs/fit/{datetime.now().strftime('%Y%m%d-%H%M%S')}'
tensorflow_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)


# Set up Early Stopping

early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

In [12]:
# Model Training

history = model.fit(
    X_train, y_train, validation_data=(X_test, y_test), epochs=100,
    callbacks = [tensorflow_callback, early_stopping_callback]
)

Epoch 1/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.8273 - loss: 0.4105 - val_accuracy: 0.8500 - val_loss: 0.3469
Epoch 2/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8553 - loss: 0.3551 - val_accuracy: 0.8568 - val_loss: 0.3625
Epoch 3/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8588 - loss: 0.3458 - val_accuracy: 0.8572 - val_loss: 0.3508
Epoch 4/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8608 - loss: 0.3412 - val_accuracy: 0.8512 - val_loss: 0.3586
Epoch 5/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8628 - loss: 0.3384 - val_accuracy: 0.8548 - val_loss: 0.3403
Epoch 6/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8632 - loss: 0.3383 - val_accuracy: 0.8580 - val_loss: 0.3494
Epoch 7/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8595 - loss: 0.3365 - val_accuracy: 0.8548 - val_loss: 0.3430
Epoch 8/100
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.8644 - loss: 0.3336 - val_accu

In [13]:
# Saving Model

model.save('model.h5')

In [ ]:
# Load Tensorboard Extension
%load_ext tensorboard

In [ ]:
%tensorboard --logdir logs/fit --port 6007